In [ ]:
import biosppy.signals as bsp

from deep_qrs_detector import DeepQRSDetector
from deep_qrs_predictor import DeepQRSPredictor
from ecg_dataset_manager import DatasetManager

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from scipy.signal import resample
from ucsd_ecg_dataset import ECGDataset

ucsd_ds = ECGDataset("./data/ucsd")

UCSD_PROCESSED_DATASET_LOCATION = './data/old_preprocessed/'
MITDB_DATASET_LOCATION = './data/mitdb/raw/'
NEW_PROCESSED_DATASET_LOCATION = './data/new_processed/'

dm = DatasetManager(UCSD_PROCESSED_DATASET_LOCATION,'',MITDB_DATASET_LOCATION, NEW_PROCESSED_DATASET_LOCATION)

database = "UCSD"
start = None
stop = None

SAMPLE_STEP = 10

if(database == "MITDB"):
    testcase = '215'
    detections_array = None #VCG

elif(database == 'UCSD'):
    testcase = 'thorntonmr_1013202209_30_07_3'
    # Split testcase name into its components
    a,b,c = testcase.split('_',2)
    detections_array = (ucsd_ds.get_trigger_detections(a,b,c) / 4.0).astype(np.int32) 

In [ ]:
run_parameters = {
    "DETECTION_CONFIDENCE_THRESHOLD": 30.0, # <-- 30 data points = 120ms
    "PEAK_COOLDOWN_THRESHOLD": 50.0 # <-- 50 data points = 200ms
}

cnn_predictor = DeepQRSPredictor('./models/cnn_predictor.h5',run_parameters['DETECTION_CONFIDENCE_THRESHOLD'],run_parameters['PEAK_COOLDOWN_THRESHOLD'])
cnn_detector = DeepQRSDetector('./models/cnn_detector.h5')

unfiltered_ecg, annotations = dm.LoadSignalAndAnnotations(testcase, database)

qrs_peaks_predictor,_ ,p_output = cnn_predictor.detect_peaks(unfiltered_ecg,return_confidence_intervals = True)
qrs_peaks_detector,_ = cnn_detector.detect_peaks(unfiltered_ecg)

hamilton_peaks = bsp.ecg.hamilton_segmenter(unfiltered_ecg, sampling_rate=250)

cnn_detector_peaks_array = np.array(qrs_peaks_detector).astype(np.int32)
cnn_predictor_peaks_array = np.array(qrs_peaks_predictor).astype(np.int32)

hamilton_array = np.array(hamilton_peaks).squeeze()
if(annotations is None):
  annotations_array = None
else:
  annotations_array = np.array(annotations)

# Interactive preview of Confidence Intervals

In [ ]:
import os
from matplotlib.lines import Line2D
from ipywidgets import interact, widgets, Layout

dataX, dataY = DeepQRSPredictor.create_windowed_dataset(cnn_predictor.SAMPLE_STEP,unfiltered_ecg,annotations)

def plot_testcase(testcase_offset):
  # Set the next testcase offset to avoid out of bounds error
  next_testcase_offset = min(testcase_offset + int(cnn_predictor.SAMPLES_PER_SLICE/cnn_predictor.SAMPLE_STEP), dataX.shape[0]-1)
  next_testcase_offset_2 = min(next_testcase_offset + int(cnn_predictor.SAMPLES_PER_SLICE/cnn_predictor.SAMPLE_STEP), dataX.shape[0]-1)
  
  # Create the plot
  plt.figure(figsize=(30,5))
  plt.plot(np.concatenate([dataX[testcase_offset,:,0], dataX[next_testcase_offset,:,0], dataX[next_testcase_offset_2,:,0]]))

  # print(p_output[testcase_offset,:])
  for i in range(0,2):
    predictionQ05 = cnn_predictor.SAMPLES_PER_SLICE - cnn_predictor.DETECTION_WINDOW + cnn_predictor.PREDICTION_WINDOW*p_output[testcase_offset,i*3+0]
    predictionX = cnn_predictor.SAMPLES_PER_SLICE - cnn_predictor.DETECTION_WINDOW + cnn_predictor.PREDICTION_WINDOW*p_output[testcase_offset,i*3+1]
    predictionQ95 = cnn_predictor.SAMPLES_PER_SLICE - cnn_predictor.DETECTION_WINDOW + cnn_predictor.PREDICTION_WINDOW*p_output[testcase_offset,i*3+2]
    if annotations_array is not None:
      trainingPredictionX = cnn_predictor.SAMPLES_PER_SLICE - cnn_predictor.DETECTION_WINDOW + cnn_predictor.PREDICTION_WINDOW*dataY[testcase_offset,i*3+1]
    # print(f"Q05: {predictionQ05} - Q95: {predictionQ95} = {predictionQ95-predictionQ05}")

    plt.axvline(x = predictionQ05, linestyle=':', color='blue')
    plt.axvline(x = predictionX, linestyle=':', color='red')
    plt.axvline(x = predictionQ95, linestyle=':', color='purple')
    # if annotations_array is not None:
    #   plt.axvline(x = trainingPredictionX, linestyle='--', color='green')
    

  plt.axvline(x = 512-64, linestyle='--', color='black') # <-- 256 millisecond before 2.048 second marker
  plt.axvline(x = 512, linestyle='-', color='black') # <-- 2.048 second marker of prediction window start

  for a in annotations_array:
    a_corrected = a - testcase_offset*cnn_predictor.SAMPLE_STEP
    if a_corrected >= 0 and a_corrected < cnn_predictor.SAMPLES_PER_SLICE*3:
      #print(f"Anno d: {a_corrected} full: {a}")
      plt.axvline(x = a_corrected, linestyle='--', color='green')

  for p in qrs_peaks_predictor:
    p_corrected = p - testcase_offset*cnn_predictor.SAMPLE_STEP
    if p_corrected >= 0 and p_corrected < cnn_predictor.SAMPLES_PER_SLICE*1.5:
      #print(f"Pred d: {p_corrected} full: {p}")
      plt.axvline(x = p_corrected, linestyle='--', color='red')
  
  # Custom legend
  if annotations_array is not None:
    custom_lines = [Line2D([0], [0], lw=1),
                Line2D([0], [0], color='blue', lw=1, linestyle=':'),
                Line2D([0], [0], color='red', lw=1, linestyle=':'),
                Line2D([0], [0], color='purple', lw=1, linestyle=':'),
                Line2D([0], [0], color='green', lw=1, linestyle='--'),
                Line2D([0], [0], color='red', lw=1, linestyle='--')]
    plt.legend(custom_lines, ['ECG Trace', r'$\Delta t_{5\%ile}$', r'$\Delta t_{50\%ile}$', r'$\Delta t_{95\%ile}$',
                              'Annotated R-peak','Predicted R-peak'], loc='lower left',fontsize=15)
  else:
    custom_lines = [Line2D([0], [0], lw=1),
                Line2D([0], [0], color='blue', lw=1, linestyle=':'),
                Line2D([0], [0], color='red', lw=1, linestyle=':'),
                Line2D([0], [0], color='purple', lw=1, linestyle=':'),
                Line2D([0], [0], color='red', lw=1, linestyle='--')]
    plt.legend(custom_lines, ['ECG Trace', r'$\Delta t_{5\%ile}$', r'$\Delta t_{50\%ile}$', r'$\Delta t_{95\%ile}$',
                              'Predicted R-peak'], loc='lower left', fontsize=15)


  # To save each plot of testcase step as a PNG
  if not os.path.exists(f'./movies/{testcase}/'):
    os.makedirs(f'./movies/{testcase}/')
  plt.savefig(f'./movies/{testcase}/frame_{testcase_offset:04d}.png')
  plt.close()

  # # To plot with interactive plot using the function itself
  # plt.show();
    

# To save each plot of testcase step as a PNG
print('frames to make:',(dataX.shape[0]-int(cnn_predictor.SAMPLES_PER_SLICE/cnn_predictor.SAMPLE_STEP)-1))
print('approximate time to make:',(dataX.shape[0]-int(cnn_predictor.SAMPLES_PER_SLICE/cnn_predictor.SAMPLE_STEP)-1)*0.13/60, 'minutes')
for testcase_offset in range(dataX.shape[0]-int(cnn_predictor.SAMPLES_PER_SLICE/cnn_predictor.SAMPLE_STEP)-1):
  plot_testcase(testcase_offset)

# # To plot with interactive plot using the function itself
# # create the slider widget with values from 0 to num_testcases-1
# testcase_slider = widgets.IntSlider(min=0, continuous_update=True,max=dataX.shape[0]-int(cnn_predictor.SAMPLES_PER_SLICE/cnn_predictor.SAMPLE_STEP)-1, 
#                                     step=1, value=0,layout=Layout(width='100%'))
# # print maximum number of slider indeces
# print('max slider value:',dataX.shape[0]-int(cnn_predictor.SAMPLES_PER_SLICE/cnn_predictor.SAMPLE_STEP)-1)
# # use the interact function to connect the slider to the plot_testcase function
# interact(plot_testcase, testcase_offset=testcase_slider);



In [ ]:
# To create a movie, type this in the terminal
'''
ffmpeg -framerate 10 -i /home/aminm/Documents/ECG\ Gating/UCSD_ECG_Clean_Version/movies/thorntonmr_1013202209_30_07_3/frame_%04d.png -c:v libx264 -vf "fps=25,format=yuv420p" /home/aminm/Documents/ECG\ Gating/UCSD_ECG_Clean_Version/movies/thorntonmr_1013202209_30_07_3/thorntonmr_1013202209_30_07_3.mp4
'''

# Generate comparison figure

In [ ]:
import matplotlib.ticker as ticker

iPlotStart = 2000 # <-- time of 8 seconds start of plot
iPlotEnd = iPlotStart+6*cnn_predictor.SAMPLES_PER_SLICE
fig = plt.figure(figsize=(20,10))

plt.subplot(411)
plt.title("VCG Trigger", fontsize = 25)
for seconds_lines in range(0,iPlotEnd-iPlotStart,250):
  plt.axvline(x=seconds_lines, linestyle=':', color='gray')
if detections_array is not None:
  plt.scatter(detections_array[(detections_array > iPlotStart) & (detections_array < iPlotEnd)]-iPlotStart,unfiltered_ecg[detections_array[(detections_array > iPlotStart) & (detections_array < iPlotEnd)]],color='red',label='Detections',zorder=3)
if(annotations_array is not None):
  for label in annotations_array[(annotations_array > iPlotStart) & (annotations_array < iPlotEnd)]:
    plt.axvline(x=(label - iPlotStart), linestyle='--', color='green',zorder=3)
plt.plot(unfiltered_ecg[iPlotStart:iPlotEnd])
plt.margins(0.01)
ax = plt.gca()
ax.set_ylim([-20000, 20000])
ax.set_yticklabels([])
ax.xaxis.set_major_locator(ticker.AutoLocator())
ax.xaxis.set_major_formatter(lambda x, pos: f"{int(x/250.0)}s")
ax.grid(False)
ax.set_facecolor('white')
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

plt.legend(bbox_to_anchor=(0, 1.02), loc="lower left",
                borderaxespad=0, ncol=3,fontsize = 18)

plt.subplot(412)
plt.title("Hamilton Detector", fontsize = 25)
for seconds_lines in range(0,iPlotEnd-iPlotStart,250):
  plt.axvline(x=seconds_lines, linestyle=':', color='gray')
plt.scatter(hamilton_array[(hamilton_array > iPlotStart) & (hamilton_array < iPlotEnd)]-iPlotStart,unfiltered_ecg[hamilton_array[(hamilton_array > iPlotStart) & (hamilton_array < iPlotEnd)]],color='red',label='Detections',zorder=3)
if(annotations_array is not None):
  for label in annotations_array[(annotations_array > iPlotStart) & (annotations_array < iPlotEnd)]:
    plt.axvline(x=(label - iPlotStart), linestyle='--', color='green',zorder=3)
plt.plot(unfiltered_ecg[iPlotStart:iPlotEnd])
plt.margins(0.01)
ax = plt.gca()
ax.set_ylim([-20000, 20000])
ax.set_yticklabels([])
ax.xaxis.set_major_locator(ticker.AutoLocator())
ax.xaxis.set_major_formatter(lambda x, pos: f"{int(x/250.0)}s")
ax.grid(False)
ax.set_facecolor('white')
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

plt.subplot(413)
plt.title("CNN Detector", fontsize = 25)
for seconds_lines in range(0,iPlotEnd-iPlotStart,250):
  plt.axvline(x=seconds_lines, linestyle=':', color='gray')
plt.scatter(cnn_detector_peaks_array[(cnn_detector_peaks_array > iPlotStart) & (cnn_detector_peaks_array < iPlotEnd)]-iPlotStart,unfiltered_ecg[cnn_detector_peaks_array[(cnn_detector_peaks_array > iPlotStart) & (cnn_detector_peaks_array < iPlotEnd)]],color='red',label='Detections',zorder=3)
if(annotations_array is not None):
  for ind,label in enumerate(annotations_array[(annotations_array > iPlotStart) & (annotations_array < iPlotEnd)]):
    if(ind == 0):
      plt.axvline(x=(label - iPlotStart), linestyle='--', color='green',label='Labels',zorder=3)
    else:
      plt.axvline(x=(label - iPlotStart), linestyle='--', color='green',zorder=3)
plt.plot(unfiltered_ecg[iPlotStart:iPlotEnd])
plt.margins(0.01)
ax = plt.gca()
ax.set_ylim([-20000, 20000])
ax.set_yticklabels([])
ax.xaxis.set_major_locator(ticker.AutoLocator())
ax.xaxis.set_major_formatter(lambda x, pos: f"{int(x/250.0)}s")
ax.grid(False)
ax.set_facecolor('white')
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

plt.subplot(414)
plt.title("CNN Predictor", fontsize = 25)
for seconds_lines in range(0,iPlotEnd-iPlotStart,250):
  plt.axvline(x=seconds_lines, linestyle=':', color='gray')
plt.scatter(cnn_predictor_peaks_array[(cnn_predictor_peaks_array > iPlotStart) & (cnn_predictor_peaks_array < iPlotEnd)]-iPlotStart,unfiltered_ecg[cnn_predictor_peaks_array[(cnn_predictor_peaks_array > iPlotStart) & (cnn_predictor_peaks_array < iPlotEnd)]],color='red',label='Detections',zorder=3)
if(annotations_array is not None):
  for ind,label in enumerate(annotations_array[(annotations_array > iPlotStart) & (annotations_array < iPlotEnd)]):
    if(ind == 0):
      plt.axvline(x=(label - iPlotStart), linestyle='--', color='green',label='Labels',zorder=3)
    else:
      plt.axvline(x=(label - iPlotStart), linestyle='--', color='green',zorder=3)
plt.plot(unfiltered_ecg[iPlotStart:iPlotEnd])
plt.margins(0.01)
ax = plt.gca()
ax.set_ylim([-20000, 20000])
ax.set_yticklabels([])
ax.xaxis.set_major_locator(ticker.AutoLocator())
ax.xaxis.set_major_formatter(lambda x, pos: f"{int(x/250.0)}s")
ax.grid(False)
ax.set_facecolor('white')
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

fig.tight_layout(pad=1.0)
plt.show()